# FoundationPose Evaluation — Colab GPU

**Ejecutar DESPUES de `00_colab_setup.ipynb`**

Este notebook:
1. Instala FoundationPose desde el repo oficial de NVIDIA
2. Descarga pesos pre-entrenados
3. Ejecuta inferencia en YCB-V y T-LESS
4. Calcula metricas BOP (ADD, ADD-S, MSSD, MSPD)
5. Guarda resultados en Google Drive

**Requiere:** GPU T4 o superior (Runtime > Change runtime type > T4 GPU)

In [ ]:
import torch
import os

assert torch.cuda.is_available(), "GPU requerida! Ve a Runtime > Change runtime type > T4 GPU"
print(f"GPU: {torch.cuda.get_device_name(0)}")

REPO_DIR = "/content/repo_tfm"
assert os.path.exists(REPO_DIR), "Ejecuta 00_colab_setup.ipynb primero"
os.chdir(REPO_DIR)

import sys
sys.path.insert(0, REPO_DIR)

## 1. Instalar FoundationPose

In [ ]:
FP_DIR = "/content/FoundationPose"

if not os.path.exists(FP_DIR):
    print("Clonando FoundationPose...")
    !git clone https://github.com/NVlabs/FoundationPose.git {FP_DIR}
    print("FoundationPose clonado")
else:
    print("FoundationPose ya existe")

# Instalar dependencias basicas
print("\nInstalando dependencias...")
!pip install -q trimesh opencv-python-headless pyrender
!pip install -q git+https://github.com/NVlabs/nvdiffrast

print("\nCompilando extensiones FoundationPose...")
!cd {FP_DIR} && bash build_all_conda.sh 2>&1 | tail -20 || echo "build_all_conda.sh completado"

# Agregar a path
sys.path.insert(0, FP_DIR)

print(f"\nFoundationPose instalado en: {FP_DIR}")
!ls -la {FP_DIR} | grep -E '(estimater|build|src)'

## 2. Descargar pesos pre-entrenados

In [ ]:
# Descargar pesos pre-entrenados de FoundationPose
WEIGHTS_DIR = "/content/drive/MyDrive/TFM/weights/foundationpose"
os.makedirs(WEIGHTS_DIR, exist_ok=True)

# Verificar si Google Drive esta montado
USE_GDRIVE = os.path.exists("/content/drive/MyDrive")

if not USE_GDRIVE:
    print("Montando Google Drive...")
    from google.colab import drive
    drive.mount('/content/drive')
    os.makedirs(WEIGHTS_DIR, exist_ok=True)

# Los pesos oficiales de FoundationPose estan en el repo de NVIDIA
# URL: https://github.com/NVlabs/FoundationPose#pre-trained-weights

print("Verificando pesos pre-entrenados...")
print(f"Directorio de pesos: {WEIGHTS_DIR}\n")

# Buscar pesos existentes
if os.path.exists(os.path.join(WEIGHTS_DIR, "model_refiner")):
    print("[OK] model_refiner encontrado")
else:
    print("[!] model_refiner NO encontrado")
    print("    Descargalo manualmente desde:")
    print("    https://github.com/NVlabs/FoundationPose#pre-trained-weights")
    print(f"    Y guardalo en: {WEIGHTS_DIR}/model_refiner\n")

if os.path.exists(os.path.join(WEIGHTS_DIR, "model_scorer")):
    print("[OK] model_scorer encontrado")
else:
    print("[!] model_scorer NO encontrado")
    print("    Descargalo manualmente desde:")
    print("    https://github.com/NVlabs/FoundationPose#pre-trained-weights")
    print(f"    Y guardalo en: {WEIGHTS_DIR}/model_scorer\n")

print("\nContenido actual:")
!ls -la {WEIGHTS_DIR} 2>/dev/null || echo "Directorio aun vacio"

## 3. Configurar evaluacion BOP

In [ ]:
# Instalar BOP Toolkit desde GitHub
print("Instalando BOP Toolkit...")
!pip install -q git+https://github.com/thodan/bop_toolkit.git

import numpy as np
import json
import time
from pathlib import Path

# Verificar datasets
DATA_DIR = f"{REPO_DIR}/data/datasets"
print(f"\nVerificando datasets en: {DATA_DIR}\n")

for ds in ['ycbv', 'tless']:
    p = Path(DATA_DIR) / ds
    if p.exists():
        models = list((p / 'models').glob('obj_*.ply')) if (p / 'models').exists() else []
        print(f"{ds.upper()}: {len(models)} modelos 3D")
        
        # Verificar imagenes de test
        test_dirs = [d for d in p.iterdir() if d.is_dir() and 'test' in d.name]
        for td in sorted(test_dirs):
            scenes = [d for d in td.iterdir() if d.is_dir()]
            print(f"  {td.name}: {len(scenes)} escenas")
    else:
        print(f"{ds.upper()}: NO encontrado (ejecuta 00_colab_setup.ipynb primero)")

## 4. Ejecutar FoundationPose en YCB-V

In [ ]:
# Importar el codigo de FoundationPose
sys.path.insert(0, FP_DIR)

# Cargar modulos de FoundationPose
from estimater import FoundationPose, ScorePredictor, PoseRefinePredictor
from datareader import BaseReader
from utils import CameraIntrinsic

# Cargar nuestro dataset loader
from src.utils.dataset_loader import BOPDataset
from tqdm.notebook import tqdm

# Cargar dataset YCB-V
print(f"Cargando YCB-V desde {DATA_DIR}/ycbv")
ycbv = BOPDataset(f"{DATA_DIR}/ycbv", split="test")
ycbv_scenes = ycbv.get_scene_ids()
print(f"YCB-V: {len(ycbv_scenes)} escenas de test\n")

# Inicializar FoundationPose
print("Inicializando FoundationPose...")
device = "cuda"

# Cargar los modelos pre-entrenados
model_refiner = PoseRefinePredictor(
    ckpt_dir=f"{WEIGHTS_DIR}/model_refiner",
    device=device,
)
model_scorer = ScorePredictor(
    ckpt_dir=f"{WEIGHTS_DIR}/model_scorer",
    device=device,
)

# Crear estimador
fm = FoundationPose(
    model_refiner=model_refiner,
    model_scorer=model_scorer,
    device=device,
)

print("FoundationPose cargado correctamente")

In [ ]:
# Ejecutar inferencia en YCB-V
results_ycbv = []
timing_total = 0
n_evaluated = 0

# Limitar para primera prueba (cambiar MAX_SCENES a None para evaluar todas)
MAX_SCENES = 5
eval_scenes = ycbv_scenes[:MAX_SCENES] if MAX_SCENES else ycbv_scenes

print(f"Evaluando {len(eval_scenes)} escenas de YCB-V\n")

for scene_id in tqdm(eval_scenes, desc="Escenas YCB-V"):
    n_images = ycbv.get_num_images(scene_id)
    scene_cameras = ycbv.load_scene_camera(scene_id)
    scene_gt = ycbv.load_scene_gt(scene_id)
    
    # Procesar hasta 50 imagenes por escena
    for img_id in range(min(n_images, 50)):
        try:
            sample = ycbv.load_sample(scene_id, img_id)
            
            rgb = sample['rgb']
            depth = sample['depth']
            mask = sample.get('mask', None)
            cam_K = sample['cam_K']
            
            # Crear intrinsicos para FoundationPose
            intrinsic = CameraIntrinsic(cam_K[0, 0], cam_K[1, 1], cam_K[0, 2], cam_K[1, 2])
            
            # Ejecutar FoundationPose
            t0 = time.time()
            
            # FoundationPose.register() devuelve las poses para cada objeto detectado
            # Se registra la escena con nuevas detecciones
            poses = fm.register(
                K=intrinsic,
                rgb=rgb,
                depth=depth,
                ob_mask=mask,  # mascara de objetos
            )
            
            elapsed = time.time() - t0
            timing_total += elapsed
            n_evaluated += 1
            
            # Guardar resultados
            for obj_id, (R, t) in poses.items():
                score = fm.model_scorer.predict_score(rgb, depth, R, t, K=intrinsic)
                results_ycbv.append({
                    'scene_id': scene_id,
                    'img_id': img_id,
                    'obj_id': int(obj_id),
                    'R_pred': R.tolist(),
                    't_pred': t.tolist(),
                    'score': float(score) if hasattr(score, 'item') else float(score),
                    'time_s': elapsed,
                })
        except Exception as e:
            print(f"\n[Error] Scene {scene_id} img {img_id}: {str(e)[:100]}")
            continue

avg_time = timing_total / max(n_evaluated, 1)
print(f"\n{'='*60}")
print(f"YCB-V: {len(results_ycbv)} predicciones en {n_evaluated} imagenes")
print(f"Tiempo promedio: {avg_time*1000:.1f} ms/imagen")
if avg_time > 0:
    print(f"FPS: {1/avg_time:.1f}")
print(f"{'='*60}")

## 5. Calcular Metricas BOP

In [ ]:
from src.perception.evaluator import PoseEvaluator
import pandas as pd

if results_ycbv:
    # Crear evaluador para YCB-V
    evaluator = PoseEvaluator(
        dataset_root=f"{DATA_DIR}/ycbv",
        dataset_name="ycbv",
    )
    
    # Calcular metricas
    print("Calculando metricas BOP (ADD, ADD-S, MSSD, MSPD)...\n")
    metrics = evaluator.evaluate(
        predictions=results_ycbv,
        metrics=['add', 'adds', 'mssd', 'mspd'],
    )
    
    # Mostrar resultados
    print(f"{'='*60}")
    print(f"FoundationPose — YCB-Video Results")
    print(f"{'='*60}")
    if isinstance(metrics, dict):
        for metric_name, value in metrics.items():
            if isinstance(value, float):
                print(f"  {metric_name:20s}: {value:.4f}")
            else:
                print(f"  {metric_name:20s}: {value}")
    else:
        print(metrics)
    print(f"{'='*60}")
else:
    print("Sin predicciones para calcular metricas")

## 6. Visualizar ejemplos

In [ ]:
import matplotlib.pyplot as plt
from src.utils.visualization import draw_pose_axes

if results_ycbv:
    # Visualizar primeros 8 resultados
    fig, axes = plt.subplots(2, 4, figsize=(20, 10))
    axes = axes.flatten()
    
    for idx in range(min(8, len(results_ycbv))):
        r = results_ycbv[idx]
        sample = ycbv.load_sample(r['scene_id'], r['img_id'])
        
        rgb = sample['rgb'].copy()
        R = np.array(r['R_pred'])
        t = np.array(r['t_pred'])
        K = sample['cam_K']
        
        # Dibujar ejes de pose en la imagen
        if hasattr(draw_pose_axes, '__call__'):
            try:
                rgb = draw_pose_axes(rgb, R, t, K, scale=0.05)
            except:
                pass
        
        axes[idx].imshow(rgb)
        axes[idx].set_title(f"S{r['scene_id']}/I{r['img_id']} Obj{r['obj_id']}\nScore={r['score']:.2f}")
        axes[idx].axis('off')
    
    plt.suptitle('FoundationPose — YCB-V Predictions', fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    print("Sin resultados para visualizar")

## 7. Ejecutar en T-LESS

In [ ]:
# Verificar si T-LESS esta disponible
tless_test_path = Path(f"{DATA_DIR}/tless/test_primesense")

if tless_test_path.exists():
    print(f"Cargando T-LESS desde {DATA_DIR}/tless")
    tless = BOPDataset(f"{DATA_DIR}/tless", split="test_primesense")
    tless_scenes = tless.get_scene_ids()
    print(f"T-LESS: {len(tless_scenes)} escenas de test\n")
    
    # Evaluar un subconjunto para prueba rapida
    results_tless = []
    timing_tless = 0
    n_tless = 0
    
    MAX_TLESS_SCENES = 3  # Primeras 3 escenas para prueba
    eval_tless_scenes = tless_scenes[:MAX_TLESS_SCENES]
    
    print(f"Evaluando {len(eval_tless_scenes)} escenas (cambiar MAX_TLESS_SCENES para mas)\n")
    
    for scene_id in tqdm(eval_tless_scenes, desc="Escenas T-LESS"):
        n_images = tless.get_num_images(scene_id)
        
        for img_id in range(min(n_images, 50)):
            try:
                sample = tless.load_sample(scene_id, img_id)
                
                rgb = sample['rgb']
                depth = sample['depth']
                mask = sample.get('mask', None)
                cam_K = sample['cam_K']
                
                intrinsic = CameraIntrinsic(cam_K[0, 0], cam_K[1, 1], cam_K[0, 2], cam_K[1, 2])
                
                t0 = time.time()
                poses = fm.register(
                    K=intrinsic,
                    rgb=rgb,
                    depth=depth,
                    ob_mask=mask,
                )
                elapsed = time.time() - t0
                timing_tless += elapsed
                n_tless += 1
                
                for obj_id, (R, t) in poses.items():
                    score = fm.model_scorer.predict_score(rgb, depth, R, t, K=intrinsic)
                    results_tless.append({
                        'scene_id': scene_id,
                        'img_id': img_id,
                        'obj_id': int(obj_id),
                        'R_pred': R.tolist(),
                        't_pred': t.tolist(),
                        'score': float(score),
                        'time_s': elapsed,
                    })
            except Exception as e:
                pass
    
    avg_time_tless = timing_tless / max(n_tless, 1)
    print(f"\n{'='*60}")
    print(f"T-LESS: {len(results_tless)} predicciones en {n_tless} imagenes")
    print(f"Tiempo promedio: {avg_time_tless*1000:.1f} ms/imagen")
    if avg_time_tless > 0:
        print(f"FPS: {1/avg_time_tless:.1f}")
    print(f"{'='*60}")
else:
    print("T-LESS no disponible.")
    print("Ejecuta 00_colab_setup.ipynb para descargarlas.")
    results_tless = []

## 8. Guardar resultados

In [ ]:
from datetime import datetime

# Determinar si usar Google Drive
USE_GDRIVE = os.path.exists('/content/drive/MyDrive')
base_dir = "/content/drive/MyDrive/TFM/experiments" if USE_GDRIVE else f"{REPO_DIR}/experiments"
os.makedirs(base_dir, exist_ok=True)

timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')

# Guardar resultados YCB-V
if results_ycbv:
    exp_dir = f"{base_dir}/foundationpose_ycbv"
    os.makedirs(exp_dir, exist_ok=True)
    
    output_ycbv = {
        'method': 'FoundationPose',
        'dataset': 'ycbv',
        'timestamp': timestamp,
        'gpu': torch.cuda.get_device_name(0),
        'n_predictions': len(results_ycbv),
        'n_images': n_evaluated,
        'avg_time_ms': avg_time * 1000,
        'fps': 1 / avg_time if avg_time > 0 else 0,
        'predictions': results_ycbv,
    }
    
    result_file_ycbv = f"{exp_dir}/results_{timestamp}.json"
    with open(result_file_ycbv, 'w') as f:
        json.dump(output_ycbv, f, indent=2)
    
    print(f"YCB-V guardado: {result_file_ycbv}")

# Guardar resultados T-LESS
if results_tless:
    exp_dir_tless = f"{base_dir}/foundationpose_tless"
    os.makedirs(exp_dir_tless, exist_ok=True)
    
    output_tless = {
        'method': 'FoundationPose',
        'dataset': 'tless',
        'timestamp': timestamp,
        'gpu': torch.cuda.get_device_name(0),
        'n_predictions': len(results_tless),
        'n_images': n_tless,
        'avg_time_ms': avg_time_tless * 1000,
        'fps': 1 / avg_time_tless if avg_time_tless > 0 else 0,
        'predictions': results_tless,
    }
    
    result_file_tless = f"{exp_dir_tless}/results_{timestamp}.json"
    with open(result_file_tless, 'w') as f:
        json.dump(output_tless, f, indent=2)
    
    print(f"T-LESS guardado: {result_file_tless}")

print(f"\nResultados persistidos en {'Google Drive' if USE_GDRIVE else 'Colab Storage'}")
print("Continua con 02_gdrnet_eval.ipynb para el baseline comparativo.")